# Experiments - Image Classification


## Setup

In [ ]:
# 1. Notebook Settings
%load_ext autoreload 
%autoreload 2

# 2. Path Setup
import sys
import os
from pathlib import Path
# Add the authors' cloned repo to the Python path so 'image', 'language', etc. can be imported
sys.path.insert(0, os.path.join(os.getcwd(), "bilinear-decomposition_(cloned_from_authors)"))


# 3. Standard Libraries
import glob
import logging
import math
import random
import re
import time
from typing import List, Optional

# 4. Data Science & Visualization
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import scipy.stats as stats
import seaborn as sns
from IPython.display import display
from codecarbon import EmissionsTracker
from einops import einsum
from tqdm import tqdm

# 5. PyTorch & Machine Learning
import torch
import torch.nn.functional as F
import torchvision
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from kornia.augmentation import RandomGaussianNoise
from torch.func import vmap
from torch.utils.data import DataLoader

# 6. Project-Specific (Image) Libraries
import image.plotting as plotting
from image import Model, MNIST, FMNIST
from image import plot_eigenspectrum, plot_explanation
from image.adversarial_attack import test_attack
from image.plotting import compare_adversarial_eigenvector_activations

# 7. Emissions Tracker
from codecarbon import EmissionsTracker

# 8. Device Configuration
torch.set_grad_enabled(False)
# device = "cuda:0" 
device = "cpu"
print(f"Using device: {device}")


## Claims:

1. **Eigenvector decomposition of bilinear MLP weights across image classification tasks reveals an interpretable low-rank structure.**
2. **Truncating lower-magnitude eigenvalue components has minimal impact on classification performance, indicating that most task-relevant informaton is captured by the leading eigenvectors.**
3. **The eigenvectors of bilinear MLPs are stable across random initializations and behave similarly across different model sizes.**
4. **Input noise regularization reduces overfitting and produces bilinear MLPs with eigenvector features that are more interpretable under eigenvalue analysis.**


### Model (+ Training) + Accuracies: MNIST DATASET

In [ ]:
# Model

tracker = EmissionsTracker(project_name="model_training_MNIST")
tracker.start()


model_mnist = Model.from_config(epochs = 20).to(device)

# Data Sets

train_mnist = MNIST(train = True, device = device)
test_mnist = MNIST(train = False, device = device)

# Train Model

metrics_mnist = model_mnist.fit(train_mnist, test_mnist)


emissions = tracker.stop()
print(f"Emissions for this block: {emissions:.4f} kg CO2eq")



In [ ]:
print("=" * 50)
print("MODEL METRICS (MNIST DATASET)")
print(metrics_mnist)

print("-" * 40)


# Average accuracy over all tasks (4 decimals)
print("\nAVERAGES (MNIST DATASET)")
print("Average Train Loss:", metrics_mnist["train/loss"].mean().round(4))
print("Average Train Accuracy:", metrics_mnist["train/acc"].mean().round(4))
print("Average Validation Loss:", metrics_mnist["val/loss"].mean().round(4))
print("Average Validation Accuracy:", metrics_mnist["val/acc"].mean().round(4))


### Model (+ Training) + Accuracies: FMNIST DATASET

In [ ]:

tracker = EmissionsTracker(project_name = "model_training_FMNIST")
tracker.start()

# Model

model_fmnist = Model.from_config(epochs = 20).to(device)

# Data Sets

train_fmnist = FMNIST(train = True, device = device)
test_fmnist = FMNIST(train = False, device = device)

# Train

metrics_fmnist = model_fmnist.fit(train_fmnist, test_fmnist)


emissions = tracker.stop()
print(f"Emissions for this block: {emissions:.4f} kg CO2eq")


In [ ]:
# Accuracies

print("=" * 50)
print("MODEL METRICS (FMNIST DATASET)")
print(metrics_fmnist)

print("-" * 40)


# Average accuracy over all tasks (4 decimals)
print("\nAVERAGES (FMNIST_DATASET)")
print("Average Train Loss:", metrics_fmnist["train/loss"].mean().round(4))
print("Average Train Accuracy:", metrics_fmnist["train/acc"].mean().round(4))
print("Average Validation Loss:", metrics_fmnist["val/loss"].mean().round(4))
print("Average Validation Accuracy:", metrics_fmnist["val/acc"].mean().round(4))


## CLAIM 1:

Eigenvector decomposition of bilinear MLP weights across image classification tasks reveals an interpretable low-rank structure.


### EIGENVECTOR - EIGENVALUE ANALYSIS (MNIST DATASET) 

In [ ]:
#Eigenvector decomposition reveals interpretable low-rank structure

for digit in range(10):
    print(f"Digit: {digit}")
    fig = plot_eigenspectrum(model_mnist, digit)
    fig.show()

In [ ]:
# Visualize the top 3 eigenvectors for each digit
vals, vecs = model_mnist.decompose()


for digit in range(10):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(f"Digit {digit} - Top 3 Eigenvectors")
    for i in range(3):
        vec = vecs[digit, -(i+1)].reshape(28, 28)
        ax = axes[i]
        im = ax.imshow(vec, cmap='RdBu', vmin=-vec.abs().max(), vmax=vec.abs().max())
        ax.axis('off')
        ax.set_title(f"Ranked Number:{i+1}")
        fig.colorbar(im, ax=ax)
    plt.show()

### EIGENVECTOR - EIGENVALUE ANALYSIS (FMNIST DATASET) 

In [ ]:
fmnist_labels = {
    0: 'T-shirt/top', 1: 'Trouser', 2: 'Pullover', 3: 'Dress', 4: 'Coat',
    5: 'Sandal', 6: 'Shirt', 7: 'Sneaker', 8: 'Bag', 9: 'Ankle boot'
}

for cls in range(10):
    print(f"Class: {fmnist_labels[cls]}")
    fig = plot_eigenspectrum(model_fmnist, cls)
    fig.show()

In [ ]:
# Visualize the top 3 eigenvectors for each digit

fmnist_labels = {
    0: 'T-shirt/top', 1: 'Trouser', 2: 'Pullover', 3: 'Dress', 4: 'Coat',
    5: 'Sandal', 6: 'Shirt', 7: 'Sneaker', 8: 'Bag', 9: 'Ankle boot'
}

vals, vecs = model_fmnist.decompose()
vecs = vecs.cpu()


for cls in range(10):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(f"{fmnist_labels[cls]} - Top 3 Eigenvectors")
    for i in range(3):
        vec = vecs[cls, -(i+1)].reshape(28, 28)
        ax = axes[i]
        im = ax.imshow(vec, cmap='RdBu', vmin=-vec.abs().max(), vmax=vec.abs().max())
        ax.axis('off')
        ax.set_title(f"Ranked Number:{i+1}")
        fig.colorbar(im, ax=ax)
    plt.show()

### OBSERVATIONS

**MNIST**

- Eigenvectors for digits such as 1, 3, 5 and 7 reveal patterns that are characteristic of those digits.
- Eigenspectrum shows decay in eigenvalue magnitude, indicating that the data is well-conditioned.

**FMNIST**
- Eigenvectors for multiple items (such as shirts, jackets and dresses) reveal outlines and patterns that are characteristic of those items.
- Eigenspectrum shows decay in eigenvalue magnitude, indicating that the data is well-conditioned. 

### CONCLUSION

- **There exists interpretability of the decomposition of the model. The model weights are not random and they encode human-understanable visual patterns.**
- **The model confirms a low-rank structure. The learnt weight matrices for each class are dominated by a few eigenvectors.**

## CLAIM 2:
**Truncating lower-magnitude eigenvalue components has minimal impact on classification performance, indicating that most task-relevant informaton is captured by the leading eigenvectors.**


### Spectral Mass Analysis: MNIST DATASET

In [ ]:
vals, vecs = model_mnist.decompose()

for digit in range(10):
    eigenvals = vals[digit].cpu().numpy()

    print(f"Digit: {digit}")
    print(f"Top 5 Eigenvalues: {eigenvals[-5:]}")
    print(f"Sum of All Eigenvalues: {eigenvals.sum():.4f}")
    print(f"Sum of Top 5 Eigenvalues: {eigenvals[-5:].sum():.4f}")
    
    total_mass = (eigenvals ** 2).sum()
    top_5_mass = (eigenvals[-5:] ** 2).sum()
    print(f"Total Mass: {total_mass:.4f}")
    print(f"Top 5 Eigenvalues mass: {top_5_mass:.4f}")
    print(f"Top 5 Contain: {top_5_mass / total_mass * 100:.2f}% of Spectal Mass")
    print("-" * 50)

### Spectral Mass Analysis: FMNIST DATASET

In [ ]:
# Create a mapping for FMNIST labels
fmnist_labels = {
    0: 'T-shirt/top', 1: 'Trouser', 2: 'Pullover', 3: 'Dress', 4: 'Coat',
    5: 'Sandal', 6: 'Shirt', 7: 'Sneaker', 8: 'Bag', 9: 'Ankle boot'
}
vals, vecs = model_fmnist.decompose()
for cls in range(10):
    eigenvals = vals[cls].cpu().numpy()
    
    print(f"Class: {fmnist_labels[cls]} ({cls})") 
    print(f"Top 5 Eigenvalues: {eigenvals[-5:]}")
    print(f"Sum of All Eigenvalues: {eigenvals.sum():.4f}")
    print(f"Sum of Top 5 Eigenvalues: {eigenvals[-5:].sum():.4f}")
    
    total_mass = (eigenvals ** 2).sum()
    top_5_mass = (eigenvals[-5:] ** 2).sum()
    print(f"Total Mass: {total_mass:.4f}")
    print(f"Top 5 Eigenvalues mass: {top_5_mass:.4f}")
    print(f"Top 5 Contain: {top_5_mass / total_mass * 100:.2f}% of Spectral Mass")
    print("-" * 50)

### TRUNCATION (MNIST DATASET)

In [ ]:
def test_truncation_mnist(model, test_data, k):

    model.eval()
    correct = 0
    total = 0

    vals, vecs = model.decompose()
    vals = vals.cpu()
    vecs = vecs.cpu()
    
    with torch.no_grad():
        for x, y in test_data:
            x_flat = x.flatten().cpu()

            scores = []

            for digit in range(10):
                idx = torch.argsort(vals[digit].abs())[-k:].flip(0)
                proj = torch.matmul(x_flat, vecs[digit, idx].T)
                activation = torch.sum(proj ** 2 * vals[digit, idx])
                scores.append(activation.item())


            pred = torch.argmax(torch.tensor(scores))

            if pred == y.cpu():
                correct += 1
            total += 1
    
    accuracy = correct / total

    return accuracy



In [ ]:
k_values = [5, 10, 15]

print('-' * 50)

for k in k_values:
    trunc_acc_mnist = test_truncation_mnist(model_mnist, test_mnist, k = k)
    avg_acc_mnist = metrics_mnist["val/acc"].mean().round(4)

    print(f"Model Accuracy: {avg_acc_mnist:.4f}")
    print(f"Top-{k} Eigenvectors Accuracy: {trunc_acc_mnist:.4f}")
    print(f"Accuracy Drop: {(avg_acc_mnist - trunc_acc_mnist):.4f}")
    print("-" * 50)

### TRUNCATION: FMNIST DATASET

In [ ]:
def test_truncation_fmnist(model, test_data, k):

    model.eval()
    correct = 0
    total = 0

    vals, vecs = model.decompose()
    vals = vals.cpu()
    vecs = vecs.cpu()
    
    with torch.no_grad():
        for x, y in test_data:
            x_flat = x.flatten().cpu()

            scores = []

            for cls in range(10):
                eigenvecs = vecs[cls]
                eigenvals = vals[cls]

                top_idx = torch.argsort(eigenvals.abs())[-k:]

                proj = torch.matmul(eigenvecs[top_idx], x_flat)

                score = torch.sum(eigenvals[top_idx] * proj ** 2)
                scores.append(score.item())

            pred = torch.argmax(torch.tensor(scores))
            if pred == y.cpu():
                correct += 1
            total += 1

    accuracy = correct / total
    return accuracy

    

In [ ]:
k_values = [5, 10, 15]

print ("-" * 50)

for k in k_values:
    trunc_acc_fmnist = test_truncation_fmnist(model_fmnist, test_fmnist, k = k)
    avg_acc_fmnist = metrics_fmnist["val/acc"].mean().round(4)

    print(f"Model Accuracy: {avg_acc_fmnist:.4f}")
    print(f"Top-{k} Eigenvectors Accuracy: {trunc_acc_fmnist:.4f}")
    print(f"Accuracy Drop: {(avg_acc_fmnist - trunc_acc_fmnist):.4f}")
    print("-" * 50)

### OBSERVATIONS

**MNIST**
- Accuracy difference between model and the truncated version:
    1) Top 5 Eigenvectors: ~2%
    2) Top 10 Eigenvectors: -1,21% (Truncated model performs way better than the original model)
    3) Top 15 Eigenvectors: -1,44% (Truncated model performs way better than the original model)

**FMNIST**
- Accuracy difference between model and the truncated version:
    1) Top 5 Eigenvectors: <1%
    2) Top 10 Eigenvectors: - 1,84% (Truncated model performs way better than the original model)
    3) Top 15 Eigenvectors: - 1,96% (Truncated model performs way better than the original model)

### CONCLUSIONS

**Not only the top eigenvectors yield the best results, but also makes the model perform better than the original model.**


 ## Setup

In [ ]:
# Initialize with restricted logging
tracker = EmissionsTracker(
    output_file="claim_3+4_emissions.csv", 
    gpu_ids=[], # Passing an empty list forces it to ignore the GPU
    log_level="error"  # roughly, to silence the errors about the GPU 
                       # (for this experiment the CPU is used)
)

## Claim 3 

The eigenvectors of bilinear MLPs are stable across random initializations and behave similarly across different model sizes.

(from the paper "Bilinear MLPs enable weight-based mechanistic interpretability":
"One important question in machine learning is whether models learn the same structure across training runs (Li et al., 2016) and across model sizes (Frankle & Carbin, 2019). In this section, we study both and find that eigenvectors are similar across runs and behave similarly across model sizes.")

In [ ]:
# dpeva_across_training_runs (= digit_position_eigenvalues_across_training_runs)
# dpeva_across_training_runs - torch.tensor in which 
#                                * the number of values in the first dimension is equal to the number of digits
#                                * the number of values in the second dimension is equal to the number of eigenvalues per training run
#                                * the number of values in the third dimension is equal to the number of training runs
# e.g., dpeva_across_training_runs[3][2][4] means something approximating "for digit 3 and for the 5th (index 4) training run, give me the 3rd (index 2) eigenvalue in the unordered list"
                                               
# dpeve_across_training_runs (= digit_position_eigenvectors_across_training_runs)
# dpeve_across_training_runs - torch.tensor in which 
#                                 * the number of values in the first dimension is equal to the number of digits
#                                 * the number of values in the second dimension is equal to the number of eigenvectors per training run
#                                 * the number of values in the third dimension is equal to the number of training runs
# e.g., dpeve_across_training_runs[5][1][2] means something approximating "for digit 5 and for the 3rd (index 2) training run, give me the 2nd (index 1) eigenvector in the unordered list"

def get_dpeva_dpeve_across_training_runs(model_size, seeds, num_training_runs=5, is_mnist=True):
    if num_training_runs != len(seeds):
        raise ValueError("The number of training runs must be equal to the number of seeds.")
    
    dpeva_across_training_runs = []
    dpeve_across_training_runs = []
    for i in range(num_training_runs):
        # Model set for 20 training epochs
        # d_hidden = the number of dimensions of the hidden layer of the model
        model = Model.from_config(d_hidden = model_size, epochs = 20, seed = seeds[i]).to(device)
        # Data Sets
        train_set = None
        test_set = None
        if is_mnist:
            train_set = MNIST(train = True, device = device)
            test_set = MNIST(train = False, device = device)
        else:
            train_set = FMNIST(train = True, device = device)
            test_set = FMNIST(train = False, device = device)

        # Train
        print(f"About to train model with d_hidden = {model_size} (training run {i+1})")
        tracker.start()
        metrics = model.fit(train_set, test_set)
        tracker.stop()
 
        # dpeva - (= digit_position_eigenvalues)
        # dpeva - torch.tensor in which
        #         * the number of values in the first dimension is equal to the number of digits
        #         * the number of values in the second dimension is equal to the number of eigenvalues
        # e.g., dpeva[8][0] means something approximating "for digit 8, give me the first (index 0) eigenvalue"

        # dpeve - (= digit_position_eigenvectors)
        # dpeve - torch.tensor in which
        #         * the number of values in the first dimension is equal to the number of digits
        #         * the number of values in the second dimension is equal to the number of eigenvectors
        # e.g., dpeve[7][3] means something approximating "for digit 7, give me the fourth (index 3) eigenvector"

        dpeva, dpeve = model.decompose()
        dpeva_across_training_runs.append(dpeva)
        dpeve_across_training_runs.append(dpeve)
    dpeva_across_training_runs = torch.stack(dpeva_across_training_runs, dim=2)    
    dpeve_across_training_runs = torch.stack(dpeve_across_training_runs, dim=2)
    return dpeva_across_training_runs, dpeve_across_training_runs

In [ ]:
def get_absolute_cosine_similarity_matrix(vectors: torch.Tensor) -> torch.Tensor:
    """
    Example:
    Input: (5, 784) - 5 vectors of dimension 784
    Output: (5, 5)  - An absolute cosine similarity matrix
    """
    # Absolute Cosine Similarity
    norms = F.normalize(vectors, p=2, dim=1)
    return torch.abs(torch.mm(norms, norms.t()))
    
vectorized_get_absolute_cosine_similarity_matrix = vmap(vmap(get_absolute_cosine_similarity_matrix))

In [ ]:
def get_cosine_similarity_matrix(vectors: torch.Tensor) -> torch.Tensor:
    """
    Example:
    Input: (5, 784) - 5 vectors of dimension 784
    Output: (5, 5)  - A cosine similarity matrix
    """
    # Cosine Similarity
    norms = F.normalize(vectors, p=2, dim=1)
    return torch.mm(norms, norms.t())
    
vectorized_get_cosine_similarity_matrix = vmap(vmap(get_cosine_similarity_matrix))

In [ ]:
def get_90_confidence_interval(data):
    """
    Computes the 90% confidence interval using the Student's t-distribution.
    """
    if len(data) < 2:
        return np.mean(data), (np.mean(data), np.mean(data))
    n = len(data)
    mean = np.mean(data)
    sem = stats.sem(data)
    interval = stats.t.interval(0.90, n-1, loc=mean, scale=sem)
    return mean, interval

In [ ]:
def get_mean_and_ci(sim_matrix):      
    # Extract upper triangle values (similarities between different runs)
    n = sim_matrix.shape[0]
    iu1 = torch.triu_indices(n, n, offset=1)
    pairwise_similarities = sim_matrix[iu1[0], iu1[1]].cpu().numpy()

    # Calculate Statistics
    # ci = confidence interval
    mean_sim, ci = get_90_confidence_interval(pairwise_similarities)
    return mean_sim, ci

In [ ]:
def get_results_for_testing_claim_3_abs_cos(model_sizes, seeds, is_mnist=True):
    
    all_means = []
    all_cis = []
    
    for model_size in model_sizes:
    
        dpeva_across_training_runs, dpeve_across_training_runs = get_dpeva_dpeve_across_training_runs(model_size, seeds, is_mnist=is_mnist)
        
        # dpabsolute_cosine_similarity_matrix.shape -> [digit][eigenvector (position)][absolute cosine similarity matrix]
        dpabsolute_cosine_similarity_matrix = vectorized_get_absolute_cosine_similarity_matrix(dpeve_across_training_runs)
        #dpeva -> [digit][eigenvalue (position)]
        dpeva = torch.mean(dpeva_across_training_runs, dim=-1)
        
        #pabsolute_cosine_similarity_matrix -> [eigenvector (position)][absolute cosine similarity matrix]
        pabsolute_cosine_similarity_matrix = torch.mean(dpabsolute_cosine_similarity_matrix, dim=0)
        #peva -> [eigenvalue (position)]
        peva = torch.mean(dpeva, dim=0)

        indices = torch.sort(torch.abs(peva), descending=True).indices
        ordered_pabsolute_cosine_similarity_matrix = pabsolute_cosine_similarity_matrix[indices]

        all_model_size_means = []
        all_model_size_cis = []
        
        for similarity_matrix in ordered_pabsolute_cosine_similarity_matrix:
            mean, ci = get_mean_and_ci(similarity_matrix)
            all_model_size_means.append(mean)
            all_model_size_cis.append(ci)

        all_means.append(all_model_size_means)
        all_cis.append(all_model_size_cis)
    
    return all_means, all_cis

In [ ]:
def get_results_for_testing_claim_3_cos(model_sizes, seeds, is_mnist=True):
    
    all_means = []
    all_cis = []
    
    for model_size in model_sizes:
    
        dpeva_across_training_runs, dpeve_across_training_runs = get_dpeva_dpeve_across_training_runs(model_size, seeds, is_mnist=is_mnist)
        
        # dpcosine_similarity_matrix.shape -> [digit][eigenvector (position)][cosine similarity matrix]
        dpcosine_similarity_matrix = vectorized_get_cosine_similarity_matrix(dpeve_across_training_runs)
        #dpeva -> [digit][eigenvalue (position)]
        dpeva = torch.mean(dpeva_across_training_runs, dim=-1)
        
        #pcosine_similarity_matrix -> [eigenvector (position)][cosine similarity matrix]
        pcosine_similarity_matrix = torch.mean(dpcosine_similarity_matrix, dim=0)
        #peva -> [eigenvalue (position)]
        peva = torch.mean(dpeva, dim=0)

        indices = torch.sort(torch.abs(peva), descending=True).indices
        ordered_pcosine_similarity_matrix = pcosine_similarity_matrix[indices]

        all_model_size_means = []
        all_model_size_cis = []
        
        for similarity_matrix in ordered_pcosine_similarity_matrix:
            mean, ci = get_mean_and_ci(similarity_matrix)
            all_model_size_means.append(mean)
            all_model_size_cis.append(ci)

        all_means.append(all_model_size_means)
        all_cis.append(all_model_size_cis)
    
    return all_means, all_cis

In [ ]:
def plot_results_for_testing_claim_3(all_means, all_cis, is_abs_cos, model_labels=None, title="Eigenvector Stability"):
    """
    creates a plot similar to the plot in Figure 5A) of the paper "Bilinear MLPs enable weight-based mechanistic interpretability"
    
    Args:
        all_means: list[list[float]] - a list in which each inner list is for a specific model
                                       and this inner list contains, for each rank, a mean
        all_cis: list[list[tuple(float, float)]] - a list in which each inner list is for a specific model and
                                                   this inner list contains, for each rank, 
                                                   the beginning and the end of a confidence interval 
        model_labels: list[str] - labels for each model (e.g., ['Model with width 128', 'Model with width 512'])
        title - the title of the plot
        is_abs_cos - whether absolute cosine similarity or cosine similarity was used in creating all_means and all_cis
    """
    plt.figure(figsize=(10, 6), dpi=100)
    
    # default labels if none provided
    if model_labels is None:
        model_labels = [f"Model {i+1}" for i in range(len(all_means))]

    # use a color map to differentiate models
    cmap = plt.cm.viridis
    num_models = len(all_means)
    colors = [cmap(i) for i in np.linspace(0, 0.9, num_models)]

    for i, (all_model_size_means, all_model_size_cis) in enumerate(zip(all_means, all_cis)):
        # ranks are 1-indexed for the plot
        ranks = np.arange(1, len(all_model_size_means) + 1)
        
        # calculate the error values relative to the mean for this specific model
        yerr_lower = [all_model_size_means[j] - all_model_size_cis[j][0] for j in range(len(all_model_size_means))]
        yerr_upper = [all_model_size_cis[j][1] - all_model_size_means[j] for j in range(len(all_model_size_means))]
        
        # create the error bar plot for this model
        plt.errorbar(
            ranks, 
            all_model_size_means, 
            yerr=[yerr_lower, yerr_upper], 
            fmt='-o', 
            color=colors[i],
            label=model_labels[i],
            elinewidth=1.5, 
            capsize=3, 
            markersize=4,
            alpha=0.8
        )
    
    # standard formatting to match scientific papers
    plt.title(title, fontsize=14, fontweight='bold', pad=20)
    plt.xlabel("Eigenvector rank (Sorted by |Eigenvalue|)", fontsize=12)
    if is_abs_cos:
        plt.ylabel("Mean |Cosine Similarity|", fontsize=12)
    else:
        plt.ylabel("Mean Cosine Similarity", fontsize=12)
    
    # Bounding to [0, 1]. Adding a tiny buffer for clarity.
    plt.ylim(0, 1.05)
    plt.xlim(0.5, 40)
    
    # adding grid for better readability of the rank values
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.legend(loc='upper right', fontsize=10)
    
    plt.tight_layout()
    plt.show()
    

### For MNIST

#### With cosine similarity

In [ ]:

model_sizes = [30, 50, 100, 300, 500, 1000]
seeds = [64, 35, 42, 41, 32]

all_means, all_cis = get_results_for_testing_claim_3_cos(model_sizes, seeds, is_mnist=True)
#~24m 7.3s

In [ ]:
plot_results_for_testing_claim_3(all_means, all_cis, is_abs_cos=False, model_labels=["30", "50", "100", "300", "500", "1000"], title="Eigenvector Stability (cosine) (MNIST)")

#### With absolute cosine similarity

In [ ]:

model_sizes = [30, 50, 100, 300, 500, 1000]
seeds = [64, 35, 42, 41, 32]

all_means, all_cis = get_results_for_testing_claim_3_abs_cos(model_sizes, seeds, is_mnist=True)
# ~29m 1.5s


In [ ]:
plot_results_for_testing_claim_3(all_means, all_cis, is_abs_cos=True, model_labels=["30", "50", "100", "300", "500", "1000"], title="Eigenvector Stability (absolute cosine) (MNIST)")

### For FMNIST

#### With cosine similarity

In [ ]:

model_sizes = [30, 50, 100, 300, 500, 1000]
seeds = [64, 35, 42, 41, 32]


all_means, all_cis = get_results_for_testing_claim_3_cos(model_sizes, seeds, is_mnist=False)
#~26 min 30 sec

In [ ]:
plot_results_for_testing_claim_3(all_means, all_cis, is_abs_cos=False, model_labels=["30", "50", "100", "300", "500", "1000"], title="Eigenvector Stability (cosine) (FMNIST)")

#### With absolute cosine similarity

In [ ]:

model_sizes = [30, 50, 100, 300, 500, 1000]
seeds = [64, 35, 42, 41, 32]

all_means, all_cis = get_results_for_testing_claim_3_abs_cos(model_sizes, seeds, is_mnist=False)
#~29m 6.4s

In [ ]:
plot_results_for_testing_claim_3(all_means, all_cis, is_abs_cos=True, model_labels=["30", "50", "100", "300", "500", "1000"], title="Eigenvector Stability (absolute cosine) (FMNIST)")

## Claim 4

Input noise regularization reduces overfitting and produces bilinear MLPs with eigenvector features that are more interpretable under eigenvalue analysis.

(from the paper "Bilinear MLPs enable weight-based mechanistic interpretability": "We found adding dense Gaussian noise to the inputs (Bricken et al., 2023a) to be an effective model regularizer, producing bilinear layers with more intuitively interpretable features.")

In [ ]:
class AddGaussianNoiseWithNorm:
    """
    To a tensor, add Gaussian noise scaled to a specific L2 norm.
    """
    def __init__(self, target_norm: float = 1.0):
        self.target_norm = target_norm

    def __call__(self, tensor: torch.Tensor) -> torch.Tensor:
        if self.target_norm <= 0:
            return tensor
            
        # Ensure noise is generated on the same device as the data
        noise = torch.randn_like(tensor)
        
        # Calculate norm across the pixel dimensions (C, H, W)
        current_norm = torch.norm(noise, p=2)
        
        # Scale to target
        scale = self.target_norm / (current_norm + 1e-8)
        return tensor + (noise * scale)

In [ ]:
def train_with_noise(train_set, validation_set, noise_value, with_norm):

    # a noise value is either a target norm or a standard deviation (std)

    # model set to 20 training epochs
    model = Model.from_config(epochs = 20).to(device)
    transform = None
    if with_norm:
        transform = AddGaussianNoiseWithNorm(target_norm=noise_value)
        print(f"About to train model with AddGaussianNoiseWithNorm(target_norm={noise_value}) applied to the training set")
    else:
        transform = RandomGaussianNoise(std=noise_value)
        print(f"About to train model with RandomGaussianNoise(std={noise_value}) applied to the training set")
    
    tracker.start()
    train_validation_metrics = model.fit(train_set, validation_set, transform=transform)
    tracker.stop()
    validation_accuracy = train_validation_metrics["val/acc"].mean().round(4)

    return model, validation_accuracy


In [ ]:
def get_results_for_testing_claim_4(train_set, validation_set, noise_values, digit, with_norm):
    if digit < 0 or digit > 9:
        return ([],[])

    validation_accuracy_per_noise_value = []
    top_processed_eigenvector_per_noise_value = []

    for noise_value in noise_values:
        model, validation_accuracy = train_with_noise(train_set, validation_set, noise_value, with_norm)
        if with_norm:
            print(f"Validation accuracy for target_norm={noise_value}: {validation_accuracy}")
        else:
            print(f"Validation accuracy for std={noise_value}: {validation_accuracy}")
        validation_accuracy_per_noise_value.append(validation_accuracy)
        eigenvalues, eigenvectors = model.decompose()
        top_processed_eigenvector = eigenvectors[digit, -1].detach().cpu().reshape(28, 28)
        top_processed_eigenvector_per_noise_value.append(top_processed_eigenvector)

    return (validation_accuracy_per_noise_value, top_processed_eigenvector_per_noise_value)

In [ ]:
def plot_results_for_testing_claim_4(noise_values, validation_accuracy_per_noise_value, top_processed_eigenvector_per_noise_value, with_norm):
    num_noise_values = len(noise_values)

    if with_norm:
        value = "Norm"
    else:
        value = "Std"
    
    # Create a figure with a top row for the line plot and a bottom row for images
    # Using GridSpec to make the layout clean
    fig = plt.figure(figsize=(15, 8))
    gs = fig.add_gridspec(2, num_noise_values)

    # 1. Plot Accuracy vs Noise Norm or Noise Std (Top Row, spanning all columns)
    ax_main = fig.add_subplot(gs[0, :])
    ax_main.plot(noise_values, validation_accuracy_per_noise_value, marker='o', linestyle='-', color='black', linewidth=2)
    ax_main.set_xlabel(f"Input Noise {value}", fontsize=12)
    ax_main.set_ylabel("Test Accuracy", fontsize=12)
    ax_main.grid(True, alpha=0.3)
    
    # Annotate points with accuracy values
    for i, acc in enumerate(validation_accuracy_per_noise_value):
        ax_main.annotate(f'{acc: .4f}', (noise_values[i], validation_accuracy_per_noise_value[i]), 
                         textcoords="offset points", xytext=(0,10), ha='center')

    # 2. Plot Eigenvector Images (Bottom Row)
    for i in range(num_noise_values):
        ax_img = fig.add_subplot(gs[1, i])
        
        eig_img = top_processed_eigenvector_per_noise_value[i]
        
        # Use a diverging color map (like 'RdBu' or 'bwr') 
        # because eigenvectors have positive and negative components
        im = ax_img.imshow(eig_img, cmap='RdBu', vmin=-eig_img.abs().max(), vmax=eig_img.abs().max())
        
        ax_img.set_title(f"{noise_values[i]}")
        ax_img.axis('off')
        
    plt.tight_layout()
    plt.suptitle("Interpretability Emerges with Noise Regularization (digit 0)", fontsize=16, y=1.02)
    plt.show()    

### For MNIST

In [ ]:
train_mnist = MNIST(train = True, device = device)
validation_mnist = MNIST(train = False, device = device)

#### With norm

In [ ]:
# with_norm=True noise_values are target_norms for AddGaussianNoiseWithNorm
# with_norm=False noise_values are stds (standard deviations) for RandomGaussianNoise

noise_values = [0.0, 0.2, 0.4, 0.6, 0.8, 1]
digit = 0
validation_accuracy_per_noise_value, top_processed_eigenvector_per_noise_value = get_results_for_testing_claim_4(train_mnist, validation_mnist, 
                                                                                                    noise_values, digit, with_norm=True)
#~4m 12.5s    

In [ ]:
plot_results_for_testing_claim_4(noise_values, validation_accuracy_per_noise_value, top_processed_eigenvector_per_noise_value, with_norm=True)

#### With standard deviation (std)

In [ ]:
noise_values = [0.0, 0.2, 0.4, 0.6, 0.8, 1]
digit = 0
validation_accuracy_per_noise_value, top_processed_eigenvector_per_noise_value = get_results_for_testing_claim_4(train_mnist, validation_mnist, 
                                                                                                    noise_values, digit, with_norm=False)

In [ ]:
plot_results_for_testing_claim_4(noise_values, validation_accuracy_per_noise_value, top_processed_eigenvector_per_noise_value, with_norm=False)

### For FMNIST

In [ ]:
train_fmnist = FMNIST(train = True, device = device)
validation_fmnist = FMNIST(train = False, device = device)

#### With norm

In [ ]:
noise_values = [0.0, 0.2, 0.4, 0.6, 0.8, 1]
digit = 0
validation_accuracy_per_noise_value, top_processed_eigenvector_per_noise_value = get_results_for_testing_claim_4(train_fmnist, validation_fmnist, 
                                                                                                    noise_values, digit, with_norm=True)
#~4m 11.3s

In [ ]:
plot_results_for_testing_claim_4(noise_values, validation_accuracy_per_noise_value, top_processed_eigenvector_per_noise_value, with_norm=True)

#### With standard deviation (std)

In [ ]:
noise_values = [0.0, 0.2, 0.4, 0.6, 0.8, 1]
digit = 0
validation_accuracy_per_noise_value, top_processed_eigenvector_per_noise_value = get_results_for_testing_claim_4(train_fmnist, validation_fmnist, 
                                                                                                    noise_values, digit, with_norm=False)
#~4m 35.8s

In [ ]:
plot_results_for_testing_claim_4(noise_values, validation_accuracy_per_noise_value, top_processed_eigenvector_per_noise_value, with_norm=False)

# Extension 1

### Setting up

In [ ]:
import os

import torch
import plotly.express as px

from image import Model, MNIST
from einops import einsum
from kornia.augmentation import RandomGaussianNoise
from image import plot_eigenspectrum, plot_explanation

from codecarbon import EmissionsTracker

# Change this to "cpu" if you don't have a GPU
device = "cpu"

# Load the MNIST dataset (directly on to a device for efficiency)
train, test = MNIST(train=True, device=device), MNIST(train=False, device=device)

### Model 1: Baseline bilinear MLP

In [ ]:
experiment_name = "baseline_bilinear_mlp"

# tracker_baseline = EmissionsTracker(
#     output_dir="./image_emissions/",
#     output_file=f"{experiment_name}_emissions.csv",
#     log_level='error'
# )

# tracker_baseline.start()

In [ ]:
models_path = f"./image_models/"

if not os.path.exists(models_path):
    os.makedirs(models_path, exist_ok=True)

save_path_baseline = models_path + f"{experiment_name}.pth"

if os.path.exists(save_path_baseline):
    model_baseline = Model.from_pretrained(save_path_baseline)
else:
    # Instantiate the model with the default configuration
    model_baseline = Model.from_config(epochs=20).to(device)
    metrics_baseline = model_baseline.fit(train, test)

    torch.save(model_baseline.state_dict(), save_path_baseline)

In [ ]:
# baseline_bilinear_mlp_emissions = tracker_baseline.stop()

In [ ]:
from image.adversarial_attack import test_attack

# Get the accuracy of the trained model on the test set during an FGSM and PGD attack

final_acc_baseline_fgsm, adversarial_examples_baseline_fgsm = test_attack(
    model=model_baseline,
    test_dataset=test,
    attack_function="FGSM"
)

final_acc_baseline_pgd, adversarial_examples_baseline_pgd = test_attack(
    model=model_baseline,
    test_dataset=test,
    attack_function="PGD"
)

In [ ]:
vals_baseline, vecs_baseline = model_baseline.decompose()
# The top positive eigenvector for the digit 0
px.imshow(vecs_baseline[0, -1].view(28, 28).cpu(), color_continuous_midpoint=0, color_continuous_scale="RdBu")

In [ ]:
# The top positive eigenvector for the digits 1-5
fig = px.imshow(vecs_baseline[1:6, -1].view(-1, 28, 28).cpu(), color_continuous_midpoint=0, color_continuous_scale="RdBu", facet_col=0)

# Hide all the abundant information to get a clean plot
fig.update_xaxes(showticklabels=False).update_yaxes(showticklabels=False)
fig.update_layout(showlegend=False).update_coloraxes(showscale=False)
fig.for_each_annotation(lambda a: a.update(text=""))

### Model 2: Regularized bilinear MLP

In [ ]:
experiment_name = "regularized_bilinear_mlp"

# tracker_regularized = EmissionsTracker(
#     output_dir="./image_emissions/",
#     output_file=f"{experiment_name}_emissions.csv",
#     log_level='error'
# )

# tracker_regularized.start()

In [ ]:
models_path = f"./image_models/"

if not os.path.exists(models_path):
    os.makedirs(models_path, exist_ok=True)

save_path_regularized = models_path + f"{experiment_name}.pth"

if os.path.exists(save_path_regularized):
    model_regularized = Model.from_pretrained(save_path_regularized)
else:
    # Instantiate the model with the default configuration
    model_regularized = Model.from_config(epochs=20).to(device)
    metrics_regularized = model_regularized.fit(train, test, transform=RandomGaussianNoise(std=0.15))

    torch.save(model_regularized.state_dict(), save_path_regularized)

In [ ]:
# tracker_regularized.stop()

In [ ]:
from image.adversarial_attack import test_attack

# Get the accuracy of the trained model on the test set during an FGSM and PGD attack

final_acc_regularized_fgsm, adversarial_examples_regularized_fgsm = test_attack(
    model=model_regularized,
    test_dataset=test,
    attack_function="FGSM"
)

final_acc_regularized_pgd, adversarial_examples_regularized_pgd = test_attack(
    model=model_regularized,
    test_dataset=test,
    attack_function="PGD"
)

In [ ]:
vals_regularized, vecs_regularized = model_regularized.decompose()
# The top positive eigenvector for the digit 0
px.imshow(vecs_regularized[0, -1].view(28, 28).cpu(), color_continuous_midpoint=0, color_continuous_scale="RdBu")

In [ ]:
# The top positive eigenvector for the digits 1-5
fig = px.imshow(vecs_regularized[1:6, -1].view(-1, 28, 28).cpu(), color_continuous_midpoint=0, color_continuous_scale="RdBu", facet_col=0)

# Hide all the abundant information to get a clean plot
fig.update_xaxes(showticklabels=False).update_yaxes(showticklabels=False)
fig.update_layout(showlegend=False).update_coloraxes(showscale=False)
fig.for_each_annotation(lambda a: a.update(text=""))

### Model 3: Adversarially trained bilinear MLP (FGSM)
Same as Model 2 plus FGSM during training

In [ ]:
experiment_name = "fgsm_regularized_bilinear_mlp"

# tracker_fgsm = EmissionsTracker(
#     output_dir="./image_emissions/",
#     output_file=f"{experiment_name}_emissions.csv",
#     log_level='error'
# )

# tracker_fgsm.start()

In [ ]:
models_path = f"./image_models/"

if not os.path.exists(models_path):
    os.makedirs(models_path, exist_ok=True)

save_path_fgsm = models_path + f"{experiment_name}.pth"

if os.path.exists(save_path_fgsm):
    model_fgsm = Model.from_pretrained(save_path_fgsm)
else:
    # Instantiate the model with the default configuration
    model_fgsm = Model.from_config(epochs=20).to(device)
    metrics_fgsm = model_fgsm.fit(train, test, transform=RandomGaussianNoise(std=0.15), defense_strategy="FGSM")

    torch.save(model_fgsm.state_dict(), save_path_fgsm)

In [ ]:
# tracker_fgsm.stop()

In [ ]:
from image.adversarial_attack import test_attack

# Get the accuracy of the trained model on the test set during an FGSM and PGD attack

final_acc_regularized_fgsm, adversarial_examples_regularized_fgsm = test_attack(
    model=model_fgsm,
    test_dataset=test,
    attack_function="FGSM"
)

final_acc_regularized_pgd, adversarial_examples_regularized_pgd = test_attack(
    model=model_fgsm,
    test_dataset=test,
    attack_function="PGD"
)

In [ ]:
vals_fgsm, vecs_fgsm = model_fgsm.decompose()
# The top positive eigenvector for the digit 0
px.imshow(vecs_fgsm[0, -1].view(28, 28).cpu(), color_continuous_midpoint=0, color_continuous_scale="RdBu")

In [ ]:
# The top positive eigenvector for the digits 1-5
fig = px.imshow(vecs_fgsm[1:6, -1].view(-1, 28, 28).cpu(), color_continuous_midpoint=0, color_continuous_scale="RdBu", facet_col=0)

# Hide all the abundant information to get a clean plot
fig.update_xaxes(showticklabels=False).update_yaxes(showticklabels=False)
fig.update_layout(showlegend=False).update_coloraxes(showscale=False)
fig.for_each_annotation(lambda a: a.update(text=""))

### Model 4: Adversarially trained bilinear MLP (PGD)
Same as Model 2 plus PGD during training

In [ ]:
experiment_name = "pgd_regularized_bilinear_mlp"

# tracker_pgd = EmissionsTracker(
#     output_dir="./image_emissions/",
#     output_file=f"{experiment_name}_emissions.csv",
#     log_level='error'
# )

# tracker_pgd.start()

In [ ]:
models_path = f"./image_models/"

if not os.path.exists(models_path):
    os.makedirs(models_path, exist_ok=True)

save_path_pgd = models_path + f"{experiment_name}.pth"

if os.path.exists(save_path_pgd):
    model_pgd = Model.from_pretrained(save_path_pgd)
else:
    # Instantiate the model with the default configuration
    model_pgd = Model.from_config(epochs=20).to(device)
    metrics_pgd = model_pgd.fit(train, test, transform=RandomGaussianNoise(std=0.15), defense_strategy="PGD")

    torch.save(model_pgd.state_dict(), save_path_pgd)

In [ ]:
# tracker_pgd.stop()

In [ ]:
from image.adversarial_attack import test_attack

# Get the accuracy of the trained model on the test set during an FGSM and PGD attack

final_acc_regularized_fgsm, adversarial_examples_regularized_fgsm = test_attack(
    model=model_pgd,
    test_dataset=test,
    attack_function="FGSM"
)

final_acc_regularized_pgd, adversarial_examples_regularized_pgd = test_attack(
    model=model_pgd,
    test_dataset=test,
    attack_function="PGD"
)

In [ ]:
vals_pgd, vecs_pgd = model_pgd.decompose()
# The top positive eigenvector for the digit 0
px.imshow(vecs_pgd[0, -1].view(28, 28).cpu(), color_continuous_midpoint=0, color_continuous_scale="RdBu")

In [ ]:
# The top positive eigenvector for the digits 1-5
fig = px.imshow(vecs_pgd[1:6, -1].view(-1, 28, 28).cpu(), color_continuous_midpoint=0, color_continuous_scale="RdBu", facet_col=0)

# Hide all the abundant information to get a clean plot
fig.update_xaxes(showticklabels=False).update_yaxes(showticklabels=False)
fig.update_layout(showlegend=False).update_coloraxes(showscale=False)
fig.for_each_annotation(lambda a: a.update(text=""))

In [ ]:
# We choose the first sample that is misclassified
adv_ex = adversarial_examples_regularized_pgd[0]

original_label = adv_ex[0]
original_img = adv_ex[2]
adv_label = adv_ex[1]
adv_img = adv_ex[3]

In [ ]:
from image.plotting import compare_adversarial_eigenvector_activations

# For an image that got misclassified during a PGD attack, we analyze the eigenvectors to discover the reason
fig = compare_adversarial_eigenvector_activations(
    model=model_pgd,
    original_img=original_img,
    adv_img=adv_img,
    original_label=original_label,
    adv_pred=adv_label,
    save_path='./pgd_adv_misclassified_analysis.png'
)

In [ ]:
import image.plotting as plotting

models = [model_baseline, model_regularized, model_fgsm, model_pgd]
model_names = ['Baseline', '+ Regularization', '+ FGSM Training', '+ PGD Training']

# We show the top positive eigenvector for 4 different digits across the different models we have trained
fig = plotting.compare_eigenvectors_across_models(
    models=models,
    model_names=model_names,
    digits=[0, 1, 2, 3, 4],
)

# Extension 2

## Image classification on CIFAR-10 and CIFAR-100
Adjusted notebook original written by *Thomas Dooms* adjusted by *Jan Jelínek*

### Codecarbon tracker

This object is going to allow us to track our emissions.

In [ ]:
tracker = EmissionsTracker(project_name="cifar100_layers_effect_experiment")
tracker.start()

### Import neccessary dependencies

In [ ]:
%load_ext autoreload
%autoreload 2

import torch
import plotly.express as px

import torchvision.transforms as transforms, torchvision, matplotlib.pyplot as plt
from image import Model

In [ ]:
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset
from datasets import load_dataset

### Define functions to train and visualize cifar-10/cifar-100 experiments

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms

def train_noise_model(
    dataset_name: str = "cifar10", 
    std: float = 0.0, 
    epochs: int = 20, 
    device: str = "cuda",
    n_layer: int = 1
):
    """
    Trains a bilinear model on either CIFAR-10 or CIFAR-100 with additive Gaussian noise.

    Args:
        dataset_name (str): "cifar10" or "cifar100".
        std (float): Standard deviation of noise to add to input images.
        epochs (int): Number of training epochs.
        device (str): Computation device.

    Returns:
        model: The trained Model instance.
    """
    dataset_name = dataset_name.lower().strip()
    
    # 1. Configure Dataset Specifics
    if dataset_name == "cifar10":
        d_output = 10
        DatasetClass = torchvision.datasets.CIFAR10
    elif dataset_name == "cifar100":
        d_output = 100
        DatasetClass = torchvision.datasets.CIFAR100
    else:
        raise ValueError("dataset_name must be 'cifar10' or 'cifar100'")

    # 2. Setup Noise Transform
    # We add random noise to the tensor
    add_noise = transforms.Lambda(lambda x: x + torch.randn_like(x) * std)


    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ])
    
    transform_noisy = transforms.Compose([
        transforms.ToTensor(),
        add_noise,
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ])

    # 3. Load Data
    print(f"Loading {dataset_name.upper()} with Noise Std: {std}...")
    trainset = DatasetClass(root="./data", train=True, download=True, transform=transform_noisy)
    # Testset using standard transform  
    testset_clean = DatasetClass(root="./data", train=False, download=True, transform=transform)

    # 4. Initialize & Train
    model = Model.from_config(epochs=epochs, d_output=d_output, d_input=3072, n_layer=n_layer).to(device)
    
    print(f"Starting training for {epochs} epochs...")
    model.fit_cifar(trainset, testset_clean)
    
    return model

### Define functions to train and visualize tiny-ImageNet experiments

In [ ]:
# 1. Create a lightweight wrapper so HF datasets behave like torchvision datasets
class HFTinyImageNetWrapper(Dataset):
    def __init__(self, hf_split, transform=None):
        self.hf_split = hf_split
        self.transform = transform

    def __len__(self):
        return len(self.hf_split)

    def __getitem__(self, idx):
        item = self.hf_split[idx]
        # Hugging Face stores images as PIL objects under the 'image' key
        image = item['image'].convert('RGB')
        label = item['label']

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
def train_noise_model_tiny_imagenet(
    std: float = 0.0,
    epochs: int = 20,
    device: str = None,
    n_layer: int = 1
):
    if device is None:
        if torch.cuda.is_available():
            device = "cuda"
        elif torch.backends.mps.is_available():
            device = "mps"
        else:
            device = "cpu"

    print(f"Using device: {device}")

    # Tiny ImageNet has 200 classes and native 64x64 images (3 * 64 * 64 = 12288)
    d_output = 200
    d_input = 12288

    # Setup Noise Transform
    add_noise = transforms.Lambda(lambda x: x + torch.randn_like(x) * std)

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ])

    transform_noisy = transforms.Compose([
        transforms.ToTensor(),
        add_noise,
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ])

    # Load Data using Hugging Face
    print(f"Loading zh-plus/tiny-imagenet with Noise Std: {std}...")
    hf_dataset = load_dataset("zh-plus/tiny-imagenet")

    # Wrap the Hugging Face dataset splits
    trainset = HFTinyImageNetWrapper(hf_dataset["train"], transform=transform_noisy)
    # Tiny ImageNet uses 'valid' for its standard evaluation split
    testset_clean = HFTinyImageNetWrapper(hf_dataset["valid"], transform=transform)

    # Initialize & Train
    model = Model.from_config(
        epochs=epochs,
        d_output=d_output,
        d_input=d_input,
        n_layer=n_layer
    ).to(device)

    print(f"Starting training for {epochs} epochs on {device}...")
    model.fit_cifar(trainset, testset_clean)

    return model

In [ ]:
import math
import plotly.express as px
from IPython.display import display

def visualize_eigenvectors(model, eigen_index: int, std: float, class_names=None):
    """
    Visualizes eigenvectors for CIFAR-10, CIFAR-100, or Tiny ImageNet.

    Args:
        model: The trained model.
        eigen_index (int): Index of eigenvector to visualize (e.g., 0 or -1).
        std (float): The noise std dev used (for title).
        class_names (list, optional): List of class names. Defaults to numbers if None.
    """
    # 1. Decompose
    vals, vecs = model.decompose()
    d_output = vecs.shape[0]  # Detects 10, 100, or 200 automatically
    d_input = vecs.shape[1]   # Total features (3072 for CIFAR, 12288 for Tiny ImageNet)

    # Dynamically calculate image spatial dimensions (e.g., 32 or 64)
    img_dim = int((d_input / 3) ** 0.5)

    # Get actual eigenvalue score
    actual_eigenvalue = vals[0, eigen_index].item()

    # Handle Class Names
    if class_names is None:
        class_names = [str(i) for i in range(d_output)]

    # 2. Determine Batching Logic
    num_batches = d_output // 10

    channels = ["Red", "Green", "Blue"]
    print(f"Generating {num_batches} figure(s) for Eigenvector {eigen_index} (Val: {actual_eigenvalue:.4f})...\n")

    # 3. Loop over batches
    for batch_idx in range(num_batches):
        start_idx = batch_idx * 10
        end_idx = start_idx + 10

        # --- A. Slice Data ---
        # Shape: (10 classes, d_input features)
        batch_vecs = vecs[start_idx:end_idx, eigen_index]

        # Reshape using the dynamic img_dim
        # Example: (10, 3, 64, 64) -> Permute to (3, 10, 64, 64) -> Flatten to (30, 64, 64)
        data = batch_vecs.view(10, 3, img_dim, img_dim).permute(1, 0, 2, 3)
        plot_data = data.reshape(30, img_dim, img_dim).cpu()

        # --- B. Create Figure ---
        title_text = (f"<b>Batch {batch_idx+1} | Classes {start_idx}-{end_idx-1} | "
                      f"EigVal: {actual_eigenvalue:.3f} | Std: {std}</b>")
        
        fig = px.imshow(plot_data,
                        facet_col=0,
                        facet_col_wrap=10,
                        color_continuous_midpoint=0,
                        color_continuous_scale="RdBu",
                        height=450, 
                        title=title_text,
                        labels={'facet_col': ''})

        # --- C. Clean Labels ---
        batch_classes = class_names[start_idx:end_idx]
        
        fig.for_each_annotation(lambda a: a.update(text=""))
        fig.update_xaxes(showticklabels=False).update_yaxes(showticklabels=False)

        # Top Labels (Classes)
        for i, name in enumerate(batch_classes):
            display_name = name[:9] + "." if len(name) > 10 else name
            fig.add_annotation(dict(x=i/9.0, y=1.1, xref="paper", yref="paper", 
                                    text=display_name, showarrow=False, font=dict(size=11)))

        # Side Labels (Channels)
        for i, name in enumerate(channels):
            fig.add_annotation(dict(x=-0.07, y=1-(i*0.45)-0.08, xref="paper", yref="paper", 
                                    text=name, showarrow=False, textangle=-90, font=dict(color="gray")))

        fig.update_layout(margin=dict(t=80, l=60))
        
        display(fig)
        if num_batches > 1:
            print("-" * 80)

Given cifar-100's more compmlicated nature, options which use more epochs and more layers have been explored

### Explore cifar-100 models

In [ ]:
import os
os.makedirs("../cifar_100_models", exist_ok=True)

In [ ]:
# tracker = EmissionsTracker(project_name="cifar100_layers_effect_experiment")
# tracker.start()

In [ ]:
model = train_noise_model(dataset_name= "cifar100", std=0, epochs=20, device="cpu")
torch.save(model.state_dict(), f"../cifar_100_models/cifar100_noise-0_epochs-2_test.pth")

In [ ]:
model = train_noise_model(dataset_name= "cifar100", std=0, epochs=20, n_layer=2, device="cpu")
torch.save(model.state_dict(), f"../cifar_100_models/cifar100_noise-0_epochs-20_layers-2.pth")

In [ ]:
model = train_noise_model(dataset_name= "cifar100", std=0, epochs=20, n_layer=3, device="cpu")
torch.save(model.state_dict(), f"../cifar_100_models/cifar100_noise-0_epochs-20_layers-3.pth")

In [ ]:
#tracker.stop()

End of experiment checking effect of layers on cifar 100

Start experiment - checking effect of layers on cifar 100 with 0.4 Gaussian noise

In [ ]:
# tracker = EmissionsTracker(project_name="cifar100_layers_effect_experiment")
# tracker.start()

In [ ]:
model = train_noise_model(dataset_name= "cifar100", std=0.4, epochs=20, device="cpu")
torch.save(model.state_dict(), f"../cifar_100_models/cifar100_noise-0.4_epochs-20.pth")

In [ ]:
model = train_noise_model(dataset_name= "cifar100", std=0.4, epochs=20, n_layer=2, device="cpu")
torch.save(model.state_dict(), f"../cifar_100_models/cifar100_noise-0.4_epochs-20_layers_2.pth")

In [ ]:
model = train_noise_model(dataset_name= "cifar100", std=0.4, epochs=20, n_layer=3, device="cpu")
torch.save(model.state_dict(), f"../cifar_100_models/cifar100_noise-0.4_epochs-20_layers_3.pth")

In [ ]:
#tracker.stop()

End of Experiment - checking effect of layers on cifar 100 with 0.4 Gaussian noise

Start of Experiment - checking of layers on cifar 100 with no noise on 100 epochs

In [ ]:
# tracker = EmissionsTracker(project_name="cifar100_layers_effect_experiment")
# tracker.start()

In [ ]:
model = train_noise_model(dataset_name= "cifar100", std=0, epochs=100, device="cpu")
torch.save(model.state_dict(), f"../cifar_100_models/cifar100_noise-0_epochs-100.pth")

In [ ]:
model = train_noise_model(dataset_name= "cifar100", std=0, epochs=100, n_layer=2, device="cpu")
torch.save(model.state_dict(), f"../cifar_100_models/cifar100_noise-0_epochs-100_layers_2.pth")

In [ ]:
model = train_noise_model(dataset_name= "cifar100", std=0, epochs=100, n_layer=3, device="cpu")
torch.save(model.state_dict(), f"../cifar_100_models/cifar100_noise-0_epochs-100_layers_3.pth")

In [ ]:
#tracker.stop()

Experiment ended earlier: models were overfitting already with 1 layer let alone more layers.

End of Experiment - checking effect of layers on cifar 100 with 0 Gaussian noise 100 epochs

Start of experiment - checking effect of layers on cifar 100 with 0.4 Gaussian noise 100 epochs

In [ ]:
# tracker = EmissionsTracker(project_name="cifar100_layers_effect_experiment")
# tracker.start()

In [ ]:
model = train_noise_model(dataset_name= "cifar100", std=0.4, epochs=100, device="cpu")
torch.save(model.state_dict(), f"../cifar_100_models/cifar100_noise-0.4_epochs-100.pth")

In [ ]:
model = train_noise_model(dataset_name= "cifar100", std=0.4, epochs=100, n_layer=2, device="cpu")
torch.save(model.state_dict(), f"../cifar_100_models/cifar100_noise-0.4_epochs-100_layers_2.pth")

In [ ]:
model = train_noise_model(dataset_name= "cifar100", std=0.4, epochs=100, n_layer=3, device="cpu")
torch.save(model.state_dict(), f"../cifar_100_models/cifar100_noise-0.4_epochs-100_layers_3.pth")

In [ ]:
#tracker.stop()

### Given Tiny ImageNet's more complicated nature (200 classes, 64x64 images), options which use more epochs and more layers are explored.

In [ ]:
os.makedirs("../tiny_imagenet_models", exist_ok=True)
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Running experiments on: {device}")



In [ ]:
tracker = EmissionsTracker(project_name="tiny_imagenet_layers_effect_experiment")
tracker.start()
model = train_noise_model_tiny_imagenet(std=0, epochs=20, device=device, n_layer=1)
torch.save(model.state_dict(), f"../tiny_imagenet_models/tiny_imagenet_noise-0_epochs-20_layers-1.pth")
model = train_noise_model_tiny_imagenet(std=0, epochs=20, device=device, n_layer=2)
torch.save(model.state_dict(), f"../tiny_imagenet_models/tiny_imagenet_noise-0_epochs-20_layers-2.pth")
model = train_noise_model_tiny_imagenet(std=0, epochs=20, device=device, n_layer=3)
torch.save(model.state_dict(), f"../tiny_imagenet_models/tiny_imagenet_noise-0_epochs-20_layers-3.pth")
tracker.stop()

End of experiment checking effect of layers on Tiny ImageNet

Start experiment - checking effect of layers on Tiny ImageNet with 0.4 Gaussian noise

In [ ]:
tracker = EmissionsTracker(project_name="tiny_imagenet_layers_effect_experiment")
tracker.start()
model = train_noise_model_tiny_imagenet(std=0.4, epochs=20, device=device, n_layer=1)
torch.save(model.state_dict(), f"../tiny_imagenet_models/tiny_imagenet_noise-0.4_epochs-20_layers-1.pth")
model = train_noise_model_tiny_imagenet(std=0.4, epochs=20, device=device, n_layer=2)
torch.save(model.state_dict(), f"../tiny_imagenet_models/tiny_imagenet_noise-0.4_epochs-20_layers-2.pth")
model = train_noise_model_tiny_imagenet(std=0.4, epochs=20, device=device, n_layer=3)
torch.save(model.state_dict(), f"../tiny_imagenet_models/tiny_imagenet_noise-0.4_epochs-20_layers-3.pth")
tracker.stop()

End of Experiment - checking effect of layers on Tiny ImageNet with 0.4 Gaussian noise

Start of Experiment - checking effect of layers on Tiny ImageNet with no noise on 100 epochs

In [ ]:
tracker = EmissionsTracker(project_name="tiny_imagenet_layers_effect_experiment")
tracker.start()
model = train_noise_model_tiny_imagenet(std=0, epochs=100, device=device, n_layer=1)
torch.save(model.state_dict(), f"../tiny_imagenet_models/tiny_imagenet_noise-0_epochs-100_layers-1.pth")
model = train_noise_model_tiny_imagenet(std=0, epochs=100, device=device, n_layer=2)
torch.save(model.state_dict(), f"../tiny_imagenet_models/tiny_imagenet_noise-0_epochs-100_layers-2.pth")
model = train_noise_model_tiny_imagenet(std=0, epochs=100, device=device, n_layer=3)
torch.save(model.state_dict(), f"../tiny_imagenet_models/tiny_imagenet_noise-0_epochs-100_layers-3.pth")
tracker.stop()

End of Experiment - checking effect of layers on Tiny ImageNet with 0 Gaussian noise 100 epochs

Start of experiment - checking effect of layers on Tiny ImageNet with 0.4 Gaussian noise 100 epochs

In [ ]:
tracker = EmissionsTracker(project_name="tiny_imagenet_layers_effect_experiment")
tracker.start()
model = train_noise_model_tiny_imagenet(std=0.4, epochs=100, device=device, n_layer=1)
torch.save(model.state_dict(), f"../tiny_imagenet_models/tiny_imagenet_noise-0.4_epochs-100_layers-1.pth")
model = train_noise_model_tiny_imagenet(std=0.4, epochs=100, device=device, n_layer=2)
torch.save(model.state_dict(), f"../tiny_imagenet_models/tiny_imagenet_noise-0.4_epochs-100_layers-2.pth")
model = train_noise_model_tiny_imagenet(std=0.4, epochs=100, device=device, n_layer=3)
torch.save(model.state_dict(), f"../tiny_imagenet_models/tiny_imagenet_noise-0.4_epochs-100_layers-3.pth")
tracker.stop()

### Train models on the CIFAR-10 dataset and visualize eigenvectors

In [ ]:
from codecarbon import EmissionsTracker
import time
# tracker = EmissionsTracker(experiment_name="CIFAR10 different stds for eigenvalues -1 and 0")
# tracker.start()

os.makedirs("../models/", exist_ok=True)

eigen_indices = [-1,0]
stds = [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1,2,5,10]
for std in stds:
    # Train model
    model = train_noise_model(dataset_name = "cifar10", std=std, epochs=20, device="cpu")
    # Save the state dictionary to a file
    torch.save(model.state_dict(), f"../models/cifar-10_std-{std}.pth")
    for eigen_index in eigen_indices:
        visualize_eigenvectors(model, eigen_index=eigen_index, std=std)
#tracker.stop()

### Train models on the CIFAR-100 without generating visualization of eigenvectors

In [ ]:
from codecarbon import EmissionsTracker
import time
# tracker = EmissionsTracker()
# tracker.start()

stds = [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1,2,5,10]
for std in stds:
    # Train model
    model = train_noise_model(dataset_name="cifar100", std=std, epochs=60, device="cpu")
    # Save the state dictionary to a file
    torch.save(model.state_dict(), f"../cifar_100_models/cifar-100_std-{std}_epoch_60.pth")
    # Visualization is not viable because the notebook becomes too large
# tracker.stop()

### Train models on the Tiny ImageNet without generating visualization of eigenvectors

In [ ]:
import os
import torch
from codecarbon import EmissionsTracker
import time

# Auto-detect device so training doesn't bottleneck on CPU
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Using device: {device}")

# Ensure the save directory exists
os.makedirs("../tiny_imagenet_models", exist_ok=True)

# tracker = EmissionsTracker()
# tracker.start()

stds = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1, 2, 5, 10]

for std in stds:
    print(f"--- Training Tiny ImageNet with std={std} ---")

    # Train model for 100 epochs using our new Tiny ImageNet function
    model = train_noise_model_tiny_imagenet(std=std, epochs=100, device=device)

    # Save the state dictionary to a file
    torch.save(model.state_dict(), f"../tiny_imagenet_models/tiny-imagenet_std-{std}_epoch_100.pth")

    # Clean up memory before the next loop
    del model
    if device == "cuda":
        torch.cuda.empty_cache()
    elif device == "mps":
        torch.mps.empty_cache()

# tracker.stop()

### Generate eigenvector plots and save them

Save plots showing eigenvectors of each class into a directory

In [ ]:
import plotly.express as px
import torch
import os
import glob
import re
import math
from typing import List, Optional

def save_eigenvector_visualization(
    model, 
    eigen_index: int, 
    std: str, 
    root_dir: str = "./visualizations",
    class_names: Optional[List[str]] = None,
    target_classes: Optional[List[str]] = None
):
    """
    Decomposes model weights and saves faceted heatmap visualizations of a specific eigenvector.
    
    Supports filtering by specific class names via `target_classes`.
    """
    # 1. Prepare Data
    vals, vecs = model.decompose()
    total_classes_in_model = vecs.shape[0]
    
    # --- Dynamic Image Dimensions for Tiny ImageNet ---
    d_input = vecs.shape[-1]
    img_dim = int((d_input / 3) ** 0.5) 
    
    # --- Resolve Full Class Names ---
    if class_names is None:
        if 'trainset' in globals() and hasattr(trainset, 'classes'):
             if len(trainset.classes) == total_classes_in_model:
                 class_names = trainset.classes
             else:
                 class_names = [str(i) for i in range(total_classes_in_model)]
        else:
            class_names = [str(i) for i in range(total_classes_in_model)]

    # --- Filter Logic ---
    if target_classes:
        indices = []
        filtered_names = []
        for target in target_classes:
            try:
                idx = class_names.index(target)
                indices.append(idx)
                filtered_names.append(target)
            except ValueError:
                print(f"Warning: Class '{target}' not found in model classes. Skipping.")
        
        if not indices:
            print("No valid target classes found. Aborting.")
            return

        vecs = vecs[torch.tensor(indices).to(vecs.device)] 
        class_names = filtered_names
        num_classes = len(class_names)
        print(f" -> Filtering for {num_classes} classes: {class_names}")
    else:
        num_classes = total_classes_in_model

    channels = ["Red", "Green", "Blue"]
    
    # --- Batching Logic ---
    batch_size = 10
    num_batches = math.ceil(num_classes / batch_size)
    
    print(f"   -> Processing Eigenvec {eigen_index} ({num_classes} classes, {num_batches} batches)...", end=" ")

    for batch_idx in range(num_batches):
        start_idx = batch_idx * batch_size
        end_idx = min(start_idx + batch_size, num_classes)
        
        # --- A. Prepare Plot Data ---
        batch_vecs = vecs[start_idx:end_idx, eigen_index]
        curr_batch_size = batch_vecs.shape[0]
        
        # Use dynamic img_dim instead of hardcoded 32x32
        data = batch_vecs.view(curr_batch_size, 3, img_dim, img_dim).permute(1, 0, 2, 3)
        plot_data = data.reshape(curr_batch_size * 3, img_dim, img_dim).cpu()
        
        # --- B. Create Figure ---
        if target_classes:
            title_text = (f"<b>Custom Selection | {', '.join(class_names[start_idx:end_idx])}<br>"
                          f"Std: {std} | Eigenvec: {eigen_index}</b>")
        else:
            # Dynamic Dataset Name
            if total_classes_in_model == 200:
                dataset_name = "Tiny ImageNet"
            elif total_classes_in_model == 100:
                dataset_name = "CIFAR-100"
            else:
                dataset_name = "CIFAR-10"
                
            title_text = (f"<b>{dataset_name} | Classes {start_idx}-{end_idx-1}<br>"
                          f"Std: {std} | Eigenvec: {eigen_index}</b>")
        
        fig = px.imshow(plot_data,
                        facet_col=0,
                        facet_col_wrap=curr_batch_size, 
                        color_continuous_midpoint=0,
                        color_continuous_scale="RdBu",
                        height=450, 
                        title=title_text,
                        labels={'facet_col': ''})


        # --- C. Clean Labels ---
        batch_class_names = class_names[start_idx:end_idx]
        fig.for_each_annotation(lambda a: a.update(text=""))
        fig.update_xaxes(showticklabels=False).update_yaxes(showticklabels=False)

        for i, name in enumerate(batch_class_names):
            display_name = name[:9] + "." if len(name) > 10 else name
            x_pos = i / (curr_batch_size - 1) if curr_batch_size > 1 else 0.5
            fig.add_annotation(dict(x=x_pos, y=1.1, xref="paper", yref="paper", 
                                    text=display_name, showarrow=False, font=dict(size=11)))

        for i, name in enumerate(channels):
            fig.add_annotation(dict(x=-0.07, y=1-(i*0.45)-0.08, xref="paper", yref="paper", 
                                    text=name, showarrow=False, textangle=-90))

        fig.update_layout(margin=dict(t=110, l=60))

        
        # --- D. Save Logic ---
        eigen_folder = f"eigenvec_{eigen_index}"
        
        if target_classes:
            safe_names = "-".join([n[:5] for n in batch_class_names]) 
            batch_folder_name = f"custom_{safe_names}"
        else:
            batch_folder_name = f"classes_{start_idx:02d}-{end_idx-1:02d}"
        
        full_save_path = os.path.join(root_dir, eigen_folder, batch_folder_name)
        os.makedirs(full_save_path, exist_ok=True)
        
        filename = f"std-{std}.png"
        fig.write_image(os.path.join(full_save_path, filename))
    
    print("Done.")


In [ ]:
import glob
import re
import torch
import torchvision.datasets as datasets
from datasets import load_dataset
from typing import List, Optional

def process_checkpoints(
    model_pattern: str,
    eigen_indices: List[int] = [-1, 0],
    save_dir: str = "./visualizations",
    target_classes: Optional[List[str]] = None
):
    """
    Scans for checkpoints and visualizes them, dynamically handling
    CIFAR-10, CIFAR-100, and Tiny ImageNet.
    """
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")

    print(f"Using device: {device}")
    print("Loading class names...")
    
    cifar10_classes = datasets.CIFAR10(root='./data', train=True, download=True).classes
    cifar100_classes = datasets.CIFAR100(root='./data', train=True, download=True).classes

    try:
        hf_ds = load_dataset("zh-plus/tiny-imagenet", split="train")
        wnids = hf_ds.features["label"].names
        import urllib.request
        import json
        url = "https://raw.githubusercontent.com/raghakot/keras-vis/master/resources/imagenet_class_index.json"
        with urllib.request.urlopen(url) as response:
            class_idx = json.loads(response.read().decode("utf-8"))
        wnid_to_name = {v[0]: v[1].replace("_", " ") for k, v in class_idx.items()}
        tiny_imagenet_classes = [wnid_to_name.get(wnid, wnid) for wnid in wnids]
    except Exception as e:
        print(f"Warning: Could not load Tiny ImageNet classes. Defaulting to numbers. Error: {e}")
        tiny_imagenet_classes = [str(i) for i in range(200)]

    files = glob.glob(model_pattern)
    parsed_files = []
    
    for f_path in files:
        # Match 'std-X' or 'noise-X'
        match = re.search(r'(?:std|noise)-(\d+(?:\.\d+)?)', f_path)
        if match:
            parsed_files.append((float(match.group(1)), f_path))

    parsed_files.sort(key=lambda x: x[0])

    if not parsed_files:
        print(f"No files found matching: {model_pattern}")
        return

    print(f"Found {len(parsed_files)} models. Processing...\n")

    for std_val, f_path in parsed_files:
        try:
            print(f"Loading model: {f_path} (Std: {std_val})")
            filename = f_path.lower()

            if "tiny-imagenet" in filename or "tiny_imagenet" in filename:
                d_out = 200
                d_in = 12288
                current_classes = tiny_imagenet_classes
            elif "cifar-100" in filename or "cifar_100" in filename:
                d_out = 100
                d_in = 3072
                current_classes = cifar100_classes
            else:
                d_out = 10
                d_in = 3072
                current_classes = cifar10_classes

            # Parse n_layer from filename (e.g., 'layers-2')
            layer_match = re.search(r'layers?-(\d+)', f_path)
            n_layer = int(layer_match.group(1)) if layer_match else 1

            # Pass n_layer to model initializer
            model = Model.from_pretrained(
                f_path,
                d_output=d_out,
                d_input=d_in,
                epochs=60,
                n_layer=n_layer
            ).to(device)

            for ev in eigen_indices:
                save_eigenvector_visualization(
                    model,
                    eigen_index=ev,
                    std=str(std_val),
                    root_dir=save_dir,
                    class_names=current_classes,
                    target_classes=target_classes
                )

            del model

            if device.type == 'cuda':
                torch.cuda.empty_cache()
            elif device.type == 'mps':
                torch.mps.empty_cache()

            print("-" * 30)
            
        except Exception as e:
            print(f"Error on {f_path}: {e}")


In [ ]:
# Process CIFAR-10 Models but ONLY visualize Truck, Horse and Automobile
process_checkpoints(
    "../models/cifar-10*.pth", 
    save_dir="./cifar_10_viz",
    target_classes=["truck", "frog", "automobile"]
)

In [ ]:
# Process CIFAR-10 Models but ONLY visualize Truck, Horse and Automobile
process_checkpoints(
    "../cifar_100_models/cifar-100*.pth", 
    save_dir="./cifar_100_viz",
    target_classes=["bowl", "boy", "bridge"] 
)

In [ ]:
# Process CIFAR-10 Models but ONLY visualize Truck, Horse and Automobile
process_checkpoints(
    "../cifar_100_models/cifar-100*.pth", 
    save_dir="./cifar_100_viz"
)

### Test accuracy of all models

In [ ]:
import sys
import os
import re
import torch
import pandas as pd
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
from typing import Optional
from datasets import load_dataset

def evaluate_models(
    models_dir: str, 
    dataset_name: str = "cifar10", 
    filter_pattern: Optional[str] = None,
    device: str = "cuda"
) -> pd.DataFrame:
    """
    Scans a directory for model checkpoints, parses metadata, and evaluates them 
    on the specified Test Set (CIFAR-10, CIFAR-100, or Tiny ImageNet).

    Args:
        models_dir (str): Path to the directory containing .pth files.
        dataset_name (str): "cifar10", "cifar100", or "tiny_imagenet". Defaults to "cifar10".
        filter_pattern (str, optional): A substring to filter files.
                                        If None, all .pth files are processed.
        device (str): "cuda", "cpu", or "mps".

    Returns:
        pd.DataFrame: Sorted DataFrame with model evaluation results.
    """
    
    # 1. Setup Test Data
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

    dataset_name = dataset_name.lower().strip()
    if dataset_name == "cifar10":
        print("Loading CIFAR-10 Test Set...")
        d_output = 10
        d_input = 3072
        testset = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)
    elif dataset_name == "cifar100":
        print("Loading CIFAR-100 Test Set...")
        d_output = 100
        d_input = 3072
        testset = torchvision.datasets.CIFAR100(root="./data", train=False, download=True, transform=transform)
    elif dataset_name == "tiny_imagenet":
        print("Loading Tiny ImageNet Test Set (from HuggingFace)...")
        d_output = 200
        d_input = 12288
        hf_dataset = load_dataset("zh-plus/tiny-imagenet")
        testset = HFTinyImageNetWrapper(hf_dataset["valid"], transform=transform)
    else:
        raise ValueError("dataset_name must be 'cifar10', 'cifar100', or 'tiny_imagenet'")

    testloader = DataLoader(testset, batch_size=2048, shuffle=False)

    # 2. Find and Filter Files
    if not os.path.exists(models_dir):
        print(f"Error: Directory not found: {models_dir}")
        return pd.DataFrame()

    all_files = os.listdir(models_dir)
    files = [f for f in all_files if f.endswith(".pth")]
    
    # Apply filter pattern if provided
    if filter_pattern:
        files = [f for f in files if filter_pattern in f]
        
    files.sort()

    print(f"Scanning directory: {models_dir}")
    print(f"Found {len(files)} models. Starting evaluation on {device}...")

    if not files:
        return pd.DataFrame()

    results = []

    # 3. Evaluation Loop
    for filename in tqdm(files, desc="Evaluating"):
        path = os.path.join(models_dir, filename)

        # A. Parse Metadata
        try:
            # Extract std (float) - handles both 'std-X' and 'noise-X' naming
            std_match = re.search(r'(?:std|noise)-([0-9\.]+)', filename)
            std = float(std_match.group(1)) if std_match else 0.0

            # Extract n_layer (int) - handles 'layers-N' naming
            layer_match = re.search(r'layers?-(\d+)', filename)
            n_layer = int(layer_match.group(1)) if layer_match else 1
            
            # Extract epoch (int) - handles both 'epoch_N' and 'epochs-N'
            epoch_match = re.search(r'epochs?[_-](\d+)', filename)
            epoch_num = int(epoch_match.group(1)) if epoch_match else 0

        except Exception:
            std, n_layer, epoch_num = -1, 1, 0

        # B. Load Model
        try:
            # Initialize model with parsed dimensions and layer count
            model = Model.from_pretrained(path, d_input=d_input, d_output=d_output, epochs=epoch_num, n_layer=n_layer).to(device)
            model.eval()
        except Exception as e:
            print(f"Error loading {filename}: {e}")
            continue

        # C. Inference
        total_loss = 0.0
        total_acc = 0.0
        num_batches = 0

        with torch.no_grad():
            for x, y in testloader:
                x, y = x.to(device), y.to(device)
                
                # Assumes model.step returns (loss, accuracy)
                loss, acc = model.step(x, y)

                total_loss += loss.item()
                total_acc += acc.item()
                num_batches += 1

        results.append({
            "filename": filename,
            "std": std,
            "n_layer": n_layer,
            "epoch": epoch_num,
            "test_acc": total_acc / num_batches,
            "test_loss": total_loss / num_batches
        })

    # 4. Return Sorted DataFrame
    df = pd.DataFrame(results)
    
    if not df.empty:
        sort_cols = [c for c in ["std", "n_layer", "epoch"] if c in df.columns]
        df = df.sort_values(by=sort_cols).reset_index(drop=True)

    return df



In [ ]:
# 1. Evaluate CIFAR-10 Models (All .pth files)
df_c10 = evaluate_models("../models", dataset_name="cifar10", device="cpu")
print(df_c10)

In [ ]:
# 2. Evaluate CIFAR-100 Models (Only epoch 60)
df_c100 = evaluate_models("../cifar_100_models", dataset_name="cifar100", filter_pattern="_epoch_60", device="cpu")
print(df_c100)

In [ ]:
# 3. Evaluate Tiny ImageNet Models
df_tiny = evaluate_models("../tiny_imagenet_models_extd", dataset_name="tiny_imagenet", device="cpu")
print(df_tiny)

In [ ]:
tracker.stop()

In [ ]:
# Process Tiny ImageNet Models and visualize specific classes
process_checkpoints(
    "../tiny_imagenet_models_extd/*tiny*.pth", 
    save_dir="./tiny_imagenet_viz_extd",
    target_classes=["goldfish", "school bus", "sports car"] 
)


